# Using Agents in LlamaIndex

An Agent is a system that leverages an AI model to interact with its environment to achieve a user-defined objective. It combines reasoning, planning, and action execution (often via external tools) to fulfil tasks.

LlamaIndex supports **three main types of reasoning agents:**

1. `Function Calling Agents` - These work with AI models that can call specific functions.
2. `ReAct Agents` - These can work with any AI that does chat or text endpoint and deal with complex reasoning tasks.
3. `Advanced Custom Agents` - These use more complex methods to deal with more complex tasks and workflows.

## Initialising Agents

To create an agent, we start by providing it with a **set of functions/tools that define its capabilities**. As of this writing, the agent will automatically use the function calling API (if available), or a standard ReAct agent loop.

In [2]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream


def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b


llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[subtract, multiply, divide, add],
    llm=llm,
    system_prompt="You are a math agent that can add, subtract, multiply, and divide numbers using provided tools.",
)

In [3]:
handler = agent.run("What is (2 + 2) * 2?")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {"a": 2, "b": 2}
```
Called tool:  add {'a': 2, 'b': 2} => 4
Thought: Now that I have the result of the addition, I need to multiply it by 2.
Action: multiply
Action Input: {'a': 4, 'b': 2}
Called tool:  multiply {'a': 4, 'b': 2} => 8
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: (2 + 2) * 2 = 8

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='(2 + 2) * 2 = 8')]), structured_response=None, current_agent_name='Agent', raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role=None, content='8', reasoning=None, tool_call_id=None, tool_calls=[]), index=0, finish_reason=None, logprobs=None, stop_reason=None)], created=1775847373, id='chatcmpl-c5b35204-4155-42e3-861b-236c10931283', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=None, object='chat.completion.chunk'), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 2, 'b': 2}, tool_id='df216071-e6ad-422c-b959-48dfc9fea7e1', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='4')], tool_name='add', raw_input={'args': (), 'kwargs': {'a': 2, 'b': 2}}, raw_output=4, is_error=False), return_direct=False), ToolCallResult(tool_name='multiply'

**Q: Why does the output start with "Thought:"?**

The agent thinks out loud before acting. It's explaining its reasoning process — it read the question "What is (2 + 2) * 2?" and decided it needs tools to solve it rather than answering directly.

---

**Q: Why did it call `add` first instead of `multiply`?**

The agent respected the order of operations (PEMDAS). The parentheses `(2 + 2)` must be solved first, so it called `add(2, 2)` before anything else.

---

**Q: What does `Called tool: add {'a': 2, 'b': 2} => 4` mean?**

It's showing you exactly what happened when the tool ran — the function name (`add`), the arguments it was given (`a=2, b=2`), and the result it returned (`4`).

---

**Q: Why does the agent "think" again after the first tool call?**

Because it's not done yet. It got `4` from the first step, but it still needs to multiply that by `2`. So it reasons through the next step before calling the second tool.

---

**Q: What does `Called tool: multiply {'a': 4, 'b': 2} => 8` mean?**

The agent passed the *result of the first tool call* (`4`) as the input to `multiply`, along with `2`. The function returned `8`. Notice the agent figured out on its own to chain the outputs — you never told it to do this.

---

**Q: What does `Answer: (2 + 2) * 2 = 8` mean?**

The agent has no more steps to run, so it gives its final response to the user. The "I can answer without using any more tools" thought right before it is the agent signalling it's done reasoning.

---

**Q: What is the big `AgentOutput(...)` block at the end?**

That's the raw Python object returned by `await handler`. It packages everything together — the final answer text, every tool call that was made, and the raw API response from the LLM. You'd use this object in code if you needed to programmatically access the answer or inspect what tools were used.

---

**Q: Why does `tool_calls` inside `AgentOutput` list both `add` and `multiply`?**

It's a complete record of every tool the agent used during the entire run, in order. Think of it as a receipt — useful for logging, debugging, or auditing what the agent did to arrive at its answer.

**Agents are stateless by default**, however, they can remember past interactions using a `Context` object. This is useful for chatbots that maintain context across multiple messages or task managers that need to track progress over time.

In [4]:
# stateless
response1 = await agent.run("What is 2 times 2?")
print(response1)
# remembering state
from llama_index.core.workflow import Context

ctx = Context(agent)

response2 = await agent.run("My name is Bob.", ctx=ctx)
print(response2)

response3 = await agent.run("What was my name again?", ctx=ctx)
print(response3)

2 times 2 is 4.
Hello, Bob! How can I assist you with math today?
Your name is Bob.


In [5]:
response3

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Your name is Bob.')]), structured_response=None, current_agent_name='Agent', raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role=None, content='.', reasoning=None, tool_call_id=None, tool_calls=[]), index=0, finish_reason=None, logprobs=None, stop_reason=None)], created=1775847434, id='chatcmpl-9dbea75a-7fef-4870-b171-3daec86f15d4', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=None, object='chat.completion.chunk'), tool_calls=[], retry_messages=[])

**Q: Why don't I see the result of "What is 2 times 2?" in the output?**

Because the output shown is only from the **last** `await agent.run(...)` call — "What was my name again?". Each `await` overwrites the `response` variable. The result of "What is 2 times 2?" was stored in `response` briefly, but then immediately replaced by the next two calls.

---

**Q: How do I fix this so I can see all three responses?**

Give each call its own variable:

```python
response1 = await agent.run("What is 2 times 2?")
print(response1)

response2 = await agent.run("My name is Bob.", ctx=ctx)
print(response2)

response3 = await agent.run("What was my name again?", ctx=ctx)
print(response3)
```

---

**Q: So the agent DID answer "What is 2 times 2?" — I just didn't capture it?**

Exactly. The agent ran, did its calculation, and returned `4`. But since the code immediately ran two more `await` calls and kept reusing the name `response`, only the final result survived.

---

**Q: Why does the last response correctly say "Your name is Bob." without me re-telling it?**

That's the `ctx` (Context) object doing its job. It remembered the earlier message "My name is Bob." and carried that memory into the third call. Without `ctx`, the agent would have no idea who Bob is.

## Creating RAG Agents with QueryEngineTools

**Agentic RAG is a powerful way to use agents to answer questions about your data.** Instead of answering questions on top of documents automatically, the agent can decide to use any other tool or flow to answer the question.

We can **wrap a `QueryEngine` as a tool** for an agent by defining a name and description — the LLM uses this information to correctly select the tool.

In [6]:
import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

# Create a vector store
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create a query engine
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm)
query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="personas",
    description="descriptions for various types of personas",
    return_direct=False,
)

# Create a RAG agent
query_engine_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[query_engine_tool],
    llm=llm,
    system_prompt="You are a helpful assistant that has access to a database containing persona descriptions. ",
)

In [8]:
handler = query_engine_agent.run(
    "Search the database for 'science fiction' and return some persona descriptions."
)
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: personas
Action Input: {"input": "science fiction"}
Called tool:  personas {'input': 'science fiction'} => Science fiction often explores themes of technology, space exploration, and futuristic societies, which can be quite distinct from the focus on cultural anthropology and the study of specific historical and societal contexts like that of Cyprus. However, if one were to draw parallels, a cultural expert might analyze how science fiction reflects or challenges cultural norms, societal structures, and human interactions in imagined future worlds. This could involve examining how different cultures might adapt to technological advancements or how fictional societies might mirror or diverge from real-world cultural practices and values.
Thought: It seems there was a misunderstanding. The observation provided does not contain persona descriptions related to science fiction

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='It seems there might be a mismatch between the query and the provided context information. The database does not have pre-defined persona descriptions specifically for science fiction. If you could provide more specific details about the type of science fiction personas you are interested in (e.g., a space explorer, a cyberpunk hacker, a time traveler), I might be able to help you craft descriptions for those.')]), structured_response=None, current_agent_name='Agent', raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role=None, content='.', reasoning=None, tool_call_id=None, tool_calls=[]), index=0, finish_reason=None, logprobs=None, stop_reason=None)], created=1775850658, id='chatcmpl-581d8c36-de90-4e45-920c-e0c7af513ae3', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None

In [9]:
print(resp.response.blocks[0].text)

It seems there might be a mismatch between the query and the provided context information. The database does not have pre-defined persona descriptions specifically for science fiction. If you could provide more specific details about the type of science fiction personas you are interested in (e.g., a space explorer, a cyberpunk hacker, a time traveler), I might be able to help you craft descriptions for those.


## What the Code is Doing

The code builds a **RAG (Retrieval-Augmented Generation) agent** — an agent that can search a database of documents before answering. Instead of the LLM relying on its own training knowledge, it first retrieves relevant stored text, then answers based on that. Here's the pipeline:

```
User Question → Agent → Query Tool → ChromaDB → Retrieved Text → LLM → Final Answer
```

Each piece plays a specific role:

- **ChromaDB** is the vector database storing your documents on disk at `./alfred_chroma_db`
- **`HuggingFaceEmbedding`** converts text into numbers (vectors) so documents can be searched by meaning, not just keywords
- **`VectorStoreIndex`** wraps ChromaDB so LlamaIndex can talk to it
- **`query_engine`** is the search interface — given a question, it finds the most relevant stored chunks
- **`QueryEngineTool`** wraps the query engine so the agent can treat it like a callable tool
- **`AgentWorkflow`** is the agent itself, which decides when and how to use the tool

---

## What Each Part of the Output Means

**The first Thought:**
```
Thought: I need to use a tool to help me answer the question.
Action: personas
Action Input: {"input": "science fiction"}
```
The agent decided it can't answer from memory alone and must search the database. It called the `personas` tool with "science fiction" as the search query.

---

**First tool result:**
```
Called tool: personas {'input': 'science fiction'} => Science fiction often explores themes of technology...
```
ChromaDB searched for documents similar to "science fiction" and returned the closest match it had — which was `persona_1.txt`, a description of a **Cypriot cultural anthropologist**. The LLM then tried to connect it to sci-fi, which produced a vague, off-topic answer.

---

**The second Thought:**
```
Thought: It seems there was a misunderstanding... I need to refine my request.
Action: personas
Action Input: {'input': 'science fiction personas'}
```
The agent noticed the first result wasn't useful and tried again with a more specific query — this is the agent self-correcting.

---

**Second tool result:**
```
Called tool: personas {'input': 'science fiction personas'} => It seems there might be a mismatch...
```
Same document came back again (`persona_1.txt`). The database simply doesn't contain any sci-fi personas, so no matter how the query is phrased, the closest match is always the Cypriot anthropologist.

---

**Final Answer:**
```
Answer: It seems there might be a mismatch... The database does not have pre-defined persona descriptions specifically for science fiction.
```
The agent gave up trying and honestly told you the database doesn't have what you asked for. This is actually correct and healthy behaviour.

---

## Warnings and Special Values to Pay Attention To

**The similarity scores:**
```
score=0.42792048433601226  (first search)
score=0.5323255300407277   (second search)
```
These are **cosine similarity scores** between your query and the stored documents. They range from 0 to 1, where 1 is a perfect match. A score of **0.43–0.53 is quite low** — it means the database found something, but it's not a good match. Anything below ~0.7 should be treated with skepticism.

**Both searches returned the exact same file:**
```
file_name: persona_1.txt  (appears in every single result)
```
This tells you your database likely only has **one document** — `persona_1.txt`. A database with one file about Cypriot culture will never answer questions about sci-fi, no matter how you phrase the query.

---

## What Data Types Are Returned

The final `resp` is an `AgentOutput` object containing:

| Field | Type | Contains |
|---|---|---|
| `response` | `ChatMessage` | The final text answer from the agent |
| `tool_calls` | `list[ToolCallResult]` | Both search attempts with inputs and outputs |
| `structured_response` | `None` | Empty — no JSON/structured output was requested |
| `raw` | `ChatCompletionStreamOutput` | The raw API response from HuggingFace |

To extract just the answer text in code:
```python
print(resp.response.blocks[0].text)
```

---

## What You Can Do Next

**The real fix is your data.** The agent worked correctly — your database is just too small and off-topic. Here's what to do:

```python
from llama_index.core import SimpleDirectoryReader

# Load actual sci-fi persona documents
documents = SimpleDirectoryReader("./your_data_folder").load_data()

# Add them to the index
for doc in documents:
    index.insert(doc)
```

Add text files describing actual personas (sci-fi, fantasy, historical, etc.) and re-run — the agent will then return meaningful results. The code and agent setup are completely correct, they just need better data to work with.

## Creating Multi-agent Systems

The `AgentWorkflow` class supports multi-agent systems. By giving each agent a name and description, the system maintains a single active speaker, with each agent able to hand off to another agent.

**Agents in LlamaIndex can also directly be used as tools** for other agents, for more complex and custom scenarios.

In [11]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
    ReActAgent,
)

# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b

# Create agent configs
# NOTE: we can use FunctionAgent or ReActAgent here.
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = ReActAgent(
    name="info_lookup",
    description="Looks up information about XYZ",
    system_prompt="Use your tool to query a RAG system to answer information about XYZ",
    tools=[query_engine_tool],
    llm=llm
)

# Create and run the workflow
agent = AgentWorkflow(
    agents=[calculator_agent, query_agent], root_agent="calculator"
)

# Run the system
response = await agent.run(user_msg="Can you add 5 and 3?")

In [12]:
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='8')]), structured_response=None, current_agent_name='calculator', raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role=None, content='8', reasoning=None, tool_call_id=None, tool_calls=[]), index=0, finish_reason=None, logprobs=None, stop_reason=None)], created=1775851385, id='chatcmpl-f62a382b-d92d-4160-9d80-c5fdda956fdd', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=None, object='chat.completion.chunk'), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 5, 'b': 3}, tool_id='41e5dd64-6a1a-4268-9e1d-f37be85dd6ca', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='8')], tool_name='add', raw_input={'args': (), 'kwargs': {'a': 5, 'b': 3}}, raw_output=8, is_error=False), return_direct=False)], retry_messages=[])

In [24]:
print(response.response.blocks[0].text)  # prints: 8
print(response.current_agent_name)        # prints: calculator

8
calculator


In [25]:
response = await agent.run(user_msg="Look up information about XYZ")
print(response.current_agent_name)  # should print: info_lookup

info_lookup


In [26]:
response = await agent.run(user_msg="What is 5 plus 3, and also tell me about XYZ?")
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='The addition of 5 plus 3 is 8. However, I do not have enough information to provide details about XYZ based on the given context.')]), structured_response=None, current_agent_name='info_lookup', raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role=None, content='.', reasoning=None, tool_call_id=None, tool_calls=[]), index=0, finish_reason=None, logprobs=None, stop_reason=None)], created=1775851853, id='chatcmpl-c8463b29-fc7f-4fc5-9656-cbf7ec8cbd78', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=None, object='chat.completion.chunk'), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 5, 'b': 3}, tool_id='4a0c5afa-8586-4c28-8501-aa49d545c1c9', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='8')], tool_name='add', raw_input={'args': (

## What the Code is Doing

This sets up a **multi-agent system** — instead of one agent with many tools, you have multiple specialized agents that can hand off tasks to each other.

```
User Message → root agent (calculator) → decides who handles it → answer
```

The two agents and their roles:

- **`calculator_agent`** — handles math, has `add` and `subtract` tools. This is the `root_agent`, meaning every conversation starts here
- **`query_agent`** — handles document lookups using the RAG tool from your previous code

The key idea is **each agent only knows about its own tools**. The calculator can't query documents, and the query agent can't do math. If a task comes in that the root agent can't handle, it hands off to the right specialist.

---

## What Each Part of the Output Means

**`current_agent_name='calculator'`**

The calculator agent handled this request from start to finish — no handoff was needed. The question "Can you add 5 and 3?" was clearly a math task, so the root agent dealt with it directly.

---

**`tool_calls`**
```python
ToolCallResult(
    tool_name='add', 
    tool_kwargs={'a': 5, 'b': 3}, 
    tool_output=ToolOutput(raw_output=8, is_error=False)
)
```
The calculator agent called `add(5, 3)`, got back `8`, and no error occurred. Only one tool was called this time — unlike the `(2+2)*2` example earlier that needed two chained calls.

---

**`response=ChatMessage(... text='8')`**

The final answer delivered to the user is simply `'8'`. Clean and minimal — the agent didn't wrap it in extra explanation.

---

**`is_error=False`**

The tool ran successfully. If this were `True`, it would mean the Python function itself crashed during execution.

---

**`retry_messages=[]`**

Empty — the agent got it right on the first try with no need to self-correct or retry.

---

## Warnings and Special Values to Pay Attention To

**`root_agent="calculator"`**

This is the most important architectural decision in the code. Every single message — regardless of content — enters through the calculator agent first. If you asked a document question, the calculator would need to recognise it can't handle it and hand off to `info_lookup`. If the root agent misjudges, the wrong agent handles the task. For production systems, you'd typically make the root agent a general router rather than a specialist.

---

**`usage=None`**

Token usage data wasn't returned by the HuggingFace streaming API. If you were using the OpenAI API, this would show you how many tokens were consumed — useful for cost tracking.

---

**`ReActAgent` vs `FunctionAgent`**

The comment in the code flags an important choice:
- `ReActAgent` — works with any LLM, uses the think-then-act loop you've seen in previous outputs
- `FunctionAgent` — only works with LLMs that support a native function-calling API, but is faster and more reliable when available

You used `ReActAgent` here which is the safer, more compatible choice with HuggingFace models.

---

## What Data Types Are Returned

The `response` variable is an `AgentOutput` object:

| Field | Type | Value here |
|---|---|---|
| `response` | `ChatMessage` | Final answer: `'8'` |
| `current_agent_name` | `str` | `'calculator'` — which agent answered |
| `tool_calls` | `list[ToolCallResult]` | One entry: `add(5,3) → 8` |
| `structured_response` | `None` | No JSON output requested |
| `retry_messages` | `list` | Empty — no retries needed |

To extract just the answer in code:
```python
print(response.response.blocks[0].text)  # prints: 8
print(response.current_agent_name)        # prints: calculator
```

---

## What You Can Do Next

**Test the handoff** — this is the most interesting thing to try next. Send a message that the calculator can't handle and see if it routes correctly to `query_agent`:

```python
response = await agent.run(user_msg="Look up information about XYZ")
print(response.current_agent_name)  # should print: info_lookup
```

**Test an ambiguous message** to see how the root agent decides:
```python
response = await agent.run(user_msg="What is 5 plus 3, and also tell me about XYZ?")
```
This is where multi-agent systems get interesting — it has to decide whether to answer both, pick one, or hand off mid-conversation.

**Stream the output** to see the agent's reasoning and any handoffs as they happen:
```python
handler = agent.run(user_msg="Can you add 5 and 3?")
async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
        print(ev.delta, end="", flush=True)
```